# Fine-tuning NomGen — Unsloth sur Google Colab

**Option B (méthode simple)**

1. **PC** — Admin NomGen → Export JSONL FR
2. **Colab** — Ce notebook (GPU T4)
3. **PC** — `ollama create` avec le fichier téléchargé

> Le fine-tuning ne se fait **pas** sur votre PC (GPU AMD). Colab fournit un GPU NVIDIA gratuit.

## Étape 0 — Activer le GPU Colab

Menu : **Runtime → Change runtime type → T4 GPU → Save**

In [1]:
# Vérifier que Colab voit bien un GPU
!nvidia-smi

Wed Jul  1 14:59:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Étape 1 — Installer Unsloth

Unsloth = bibliothèque simple pour fine-tuner un LLM et exporter vers Ollama.

In [2]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

## Étape 2 — Uploader votre JSONL NomGen

Dans NomGen : **Admin → Datasets → JSONL FR** → télécharger `nomgen_finetuning_fr.jsonl`

Puis exécutez la cellule ci-dessous et choisissez le fichier.

In [3]:
from google.colab import files
uploaded = files.upload()  # sélectionnez nomgen_finetuning_fr.jsonl
JSONL_PATH = list(uploaded.keys())[0]
print(f"Fichier chargé : {JSONL_PATH}")

Saving nomgen_training_all (1).jsonl to nomgen_training_all (1).jsonl
Fichier chargé : nomgen_training_all (1).jsonl


## Étape 3 — Charger le modèle Qwen2.5

On utilise **3B** (plus rapide sur Colab gratuit). Pour 7B, changez le nom du modèle.

In [4]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"  # ou "unsloth/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Modèle prêt ")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Modèle prêt 


## Étape 4 — Préparer le dataset NomGen

Conversion du format exporté par BrandForge (`messages`) vers le format chat Qwen.

In [5]:
import json
from datasets import Dataset

rows = []
with open(JSONL_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        messages = obj["messages"]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        rows.append({"text": text})

dataset = Dataset.from_list(rows)
print(f"Exemples d'entraînement : {len(dataset)}")
print("--- Aperçu ---")
print(dataset[0]["text"][:500])

Exemples d'entraînement : 18
--- Aperçu ---
<|im_start|>system
Tu es un expert en naming. Génère des noms de marque dans le secteur LUXE. Réponds en JSON uniquement.<|im_end|>
<|im_start|>user
Contexte : marque de luxe pour parfums
Génère des noms de marque secteur LUXE.<|im_end|>
<|im_start|>assistant
["Veloria"]<|im_end|>



## Étape 5 — Entraîner (fine-tuning LoRA)

In [6]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=100,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer.train()
print("Entraînement terminé ✓")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/18 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18 | Num Epochs = 34 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,3.698690
20,1.921605
30,1.469495
40,0.939231
50,0.544276
60,0.221739
70,0.089309
80,0.039935
90,0.033950
100,0.029550


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.


Entraînement terminé ✓


## Étape 6 — Tester le modèle (optionnel)

In [7]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Tu es un expert en naming. Génère des noms de marque dans le secteur tech. Réponds en JSON uniquement."},
    {"role": "user", "content": "Contexte : startup tech écologique\nGénère des noms de marque secteur tech."},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.8)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

system
Tu es un expert en naming. Génère des noms de marque dans le secteur tech. Réponds en JSON uniquement.
user
Contexte : startup tech écologique
Génère des noms de marque secteur tech.
assistant
["EcoLink", "NaturaLink", "Ecospeak", "LinkEco", "LinkNatura"]


## Étape 7 — Exporter pour Ollama (GGUF + Modelfile)

Téléchargez le ZIP sur votre PC.

In [8]:
EXPORT_DIR = "nomgen_qwen_finetuned"
model.save_pretrained_gguf(EXPORT_DIR, tokenizer, quantization_method="q4_k_m")
print("Export GGUF terminé ✓")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in nomgen_qwen_finetuned/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:07<00:00, 63.71s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:33<00:00, 46.76s/it]


Unsloth: Merge process complete. Saved to `/content/nomgen_qwen_finetuned`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9827-mix-1f1aaa4 (app-b9827-mix-1f1aaa4-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['nomgen_qwen_finetuned_gguf/qwen2.5-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions 

In [9]:
# Créer un ZIP pour téléchargement
!apt-get install -y zip > /dev/null
!zip -r nomgen_qwen_ollama.zip {EXPORT_DIR}

from google.colab import files
files.download("nomgen_qwen_ollama.zip")

  adding: nomgen_qwen_finetuned/ (stored 0%)
  adding: nomgen_qwen_finetuned/tokenizer_config.json (deflated 83%)
  adding: nomgen_qwen_finetuned/config.json (deflated 77%)
  adding: nomgen_qwen_finetuned/tokenizer.json (deflated 81%)
  adding: nomgen_qwen_finetuned/generation_config.json (deflated 33%)
  adding: nomgen_qwen_finetuned/chat_template.jinja (deflated 71%)
  adding: nomgen_qwen_finetuned/.cache/ (stored 0%)
  adding: nomgen_qwen_finetuned/.cache/huggingface/ (stored 0%)
  adding: nomgen_qwen_finetuned/.cache/huggingface/CACHEDIR.TAG (deflated 24%)
  adding: nomgen_qwen_finetuned/.cache/huggingface/.gitignore (stored 0%)
  adding: nomgen_qwen_finetuned/.cache/huggingface/download/ (stored 0%)
  adding: nomgen_qwen_finetuned/.cache/huggingface/download/model-00002-of-00002.safetensors.metadata (deflated 32%)
  adding: nomgen_qwen_finetuned/.cache/huggingface/download/model.safetensors.index.json.metadata (deflated 26%)
  adding: nomgen_qwen_finetuned/.cache/huggingface/downl

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Étape 8 — Sur votre PC (Windows + Ollama)

1. Dézipper `nomgen_qwen_ollama.zip`
2. Ouvrir PowerShell dans le dossier décompressé
3. Exécuter :

```powershell
ollama create nomgen-qwen25 -f Modelfile
ollama run nomgen-qwen25
```

4. Dans NomGen → **Mode B** → modèle Ollama local (`qwen2.5` ou votre modèle custom si configuré)

---

**C'est fini.** Vous avez fine-tuné NomGen sans LLaMA-Factory.